# Template: Yearly-Mean Linear Regression (Python)

Copy for any panel → yearly aggregate → OLS trend → future projection task.

Edit only the config cell.

## Config

In [ ]:
DATA_PATH    = "data/foodproduction.csv"
TIME_COL     = "year"
TARGET_COL   = "totalprod"
GROUP_AGG    = "mean"
FUTURE_START = 2013
FUTURE_END   = 2050
NOISE_FRAC   = 0.12
N_SIMS       = 300
N_BOOT       = 500
SEED         = 42

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

plt.style.use("seaborn-v0_8-whitegrid")
df = pd.read_csv(DATA_PATH)
print(df.shape)
display(df.head())

In [ ]:
agg = df.groupby(TIME_COL)[TARGET_COL].agg(GROUP_AGG).reset_index()
X = agg[TIME_COL].values.reshape(-1, 1)
y = agg[TARGET_COL].values
regr = LinearRegression().fit(X, y)
print(f"Slope {regr.coef_[0]:.4g} | Intercept {regr.intercept_:.4g} | R² {regr.score(X,y):.4f}")
y_pred = regr.predict(X)

In [ ]:
X_fut = np.arange(FUTURE_START, FUTURE_END+1).reshape(-1, 1)
y_fut = regr.predict(X_fut)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(X, y, s=50, edgecolor="k", label="Observed")
ax.plot(X, y_pred, lw=2, label="OLS fit")
ax.plot(X_fut, y_fut, lw=2, ls="--", label=f"→ {FUTURE_END}")
ax.axhline(0, color="gray", ls=":", alpha=0.6)
ax.set_xlabel(TIME_COL); ax.set_ylabel(TARGET_COL); ax.legend()
plt.tight_layout(); plt.show()
print(f"{FUTURE_END} pred: {y_fut[-1]:,.0f}")

In [ ]:
np.random.seed(SEED)
slopes = [LinearRegression().fit(X, y + np.random.normal(0, y.std()*NOISE_FRAC, len(y))).coef_[0]
          for _ in range(N_SIMS)]
print(f"Noise slopes — mean {np.mean(slopes):.0f}, 95% [{np.percentile(slopes,2.5):.0f}, {np.percentile(slopes,97.5):.0f}]")

preds = []
idx = np.arange(len(y))
for _ in range(N_BOOT):
    s = np.random.choice(idx, size=len(idx), replace=True)
    preds.append(LinearRegression().fit(X[s], y[s]).predict([[FUTURE_END]])[0])
lo, hi = np.percentile(preds, [2.5, 97.5])
print(f"{FUTURE_END} bootstrap 95% CI: [{lo:,.0f}, {hi:,.0f}]")

## Audience checklist
- [ ] Data literacy? (high → R² + CI; low → headline + chart)
- [ ] Subject expertise? (expert → skip definitions)
- [ ] Executive skim? → put the key number first
- [ ] Technical review? → appendix with code & diagnostics